In [1]:
import MDAnalysis as mda
import mdtraj as md
import os, sys, glob
import numpy as np

from openmm import *
from openmm.app import *
from openmm.unit import *


from chimpss.bridgeport.forcefield import ForceFieldHandler as ff_handler
from chimpss.bridgeport.ligand import Ligand
from chimpss.bridgeport.openmm_joiner import Joiner

from chimpss.motorrow import MotorRow
from datetime import datetime

In [2]:
bridgeport_pdb = '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.pdb'
bridgeport_xml = '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/systems/6drx_h8g.xml'
opm_pdb =        '/media/volume/Josephs-Volume/githubs/Bridgeport/5HT2B/input_pdb/opm_4ib4.pdb'

In [2]:
def parameterize_from_pdb(pdb_file, nonbonded_method, nonbonded_cutoff=None,
                          lig_resname='UNK', cleanup=True, build_w_obc2=False):

    #Seperate the stuff
    uni = mda.Universe(pdb_file)
    print(len(uni.atoms))
    not_lig = uni.select_atoms(f'not resname {lig_resname}')
    lig = uni.select_atoms('resname UNK')
    not_lig.write('dummy_P.pdb')
    lig.write('dummy_L.pdb')

    #Get the Parameters
    ff_xmls = ['amber14/protein.ff14SB.xml',
               'amber14/lipid17.xml',
               '/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/wat_opc3.xml'] #That bridgeport directory from the import cells needs to be in path?

    if build_w_obc2:
       ff_xmls.append('implicit/obc2.xml')
    
    ff = ForceField(*ff_xmls)
    pdb = PDBFile('dummy_P.pdb')
    rec_top, rec_pos = pdb.getTopology(), pdb.getPositions()
    
    if nonbonded_cutoff is None:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method)
    elif type(nonbonded_cutoff) == int or type(nonbonded_cutoff) == float:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method, nonbondedCutoff=Quantity(value=nonbonded_cutoff, unit=nanometer))
    elif type(nonbonded_cutoff) == Quantity:
        rec_sys = ff.createSystem(rec_top, nonbondedMethod=nonbonded_method, nonbondedCutoff=nonbonded_cutoff)
    
    #Ligand Too
    ligand = Ligand(working_dir='./', name='dummy_L', resname='UNK', smiles="CCN(CC)C(=O)N[C@@H]1CN([C@@H]2Cc3c[nH]c4c3c(ccc4)C2=C1)C")
    ligand.prepare_ligand()
    lig_sys, lig_top, lig_pos = ff_handler.ForceFieldHandler('dummy_L.sdf').main(use_pme='NoCutoff')
    
    #Joiner
    sys, top, pos = Joiner((lig_sys, lig_top, lig_pos), (rec_sys, rec_top, rec_pos)).main()
    print(sys.getNumParticles())
    for force in sys.getForces():
        if hasattr(force, 'getNonbondedMethod'):
            print(force)
            print(force.getNonbondedMethod(), force.getNumParticles())
            
    if cleanup:
        os.remove('dummy_P.pdb')
        os.remove('dummy_L.pdb')
        os.remove('dummy_L.sdf')
    
    return sys, top, pos

In [3]:
def implicit_membrane_force(system, nonbonded_cutoff=None, nonbonded_method=0, water_pore=True):
    #4pi = 12.56637
    """
    Just build the force, doesn't add it to the system
    nonbonded_methods 0 = NoCutoff, 1=CutoffNonPeriodic, 2=CutoffPeriodic
    """

    im_mem = CustomGBForce()
    #Dielectric Parameters
    _ = im_mem.addGlobalParameter('epsWater', 78.3) #water
    _ = im_mem.addGlobalParameter('epsMid', 3) #mid
    _ = im_mem.addGlobalParameter('epsCore', 1) #inner
    _ = im_mem.addGlobalParameter("soluteDielectric", 1) #Solute
    #Membrane Slab Parameters
    _ = im_mem.addGlobalParameter('thickness1', 1.2) #Inner Slab
    _ = im_mem.addGlobalParameter('thickness2', 1.8) #middle slab
    _ = im_mem.addGlobalParameter('alpha1', 7.2) #sig aggresiveness 
    _ = im_mem.addGlobalParameter('alpha2', 7.2) #sig aggresiveness
    #Per Particle Parameters
    _ = im_mem.addPerParticleParameter("q")
    _ = im_mem.addPerParticleParameter("radius")
    _ = im_mem.addPerParticleParameter("scale")
    #Pore Dependent Parameters
    if water_pore:
        _ = im_mem.addGlobalParameter('Pore_Radius', 0.75) #Nanometers from z axis to inflection point 
        _ = im_mem.addGlobalParameter('Pore_alpha', 21.6) 
    
    
    # See https://docs.openmm.org/latest/userguide/theory/02_standard_forces.html#gbsaobcforce
    # See 
    
    #This is allegedly something called the Hawkins-Cramer-Truhlar descreening function
    _ = im_mem.addComputedValue("Imol", "step(r+sr2-or1)*0.5*(1/L-1/U+0.25*(1/U^2-1/L^2)*(r-sr2*sr2/r)+0.5*log(L/U)/r+C);"
                                "U=r+sr2;" 
                                "C=2*(1/or1-1/L)*step(sr2-r-or1);" 
                                "L=max(or1, D);" 
                                "D=abs(r-sr2);" 
                                "sr2 = scale2*or2;"
                                "or1=radius1-0.009;"
                                "or2=radius2-0.009", CustomGBForce.ParticlePairNoExclusions)

    #Define the outside dielectric
    if water_pore:
        #As a function of X, Y, Z
        _ = im_mem.addComputedValue('eps', 'epsWater + (epsZ - epsWater)*radialsig;'
                                    'epsZ = epsCore + (epsMid - epsCore)*sigmoid1 + (epsWater-epsMid)*sigmoid2;'
                                    'radialsig=1/(1+exp(-1*Pore_alpha*(abs(sqrt(x^2 + y^2))-Pore_Radius)));'
                                    'sigmoid1=1/(1+exp(-1*alpha1*(abs(z)-thickness1)));'
                                    'sigmoid2=1/(1+exp(-1*alpha2*(abs(z)-thickness2)))', CustomGBForce.SingleParticle)
    else:
        #As a Function of Z
        _ = im_mem.addComputedValue('eps', 'epsCore + (epsMid - epsCore)*sigmoid1 + (epsWater-epsMid)*sigmoid2;'
                                    'sigmoid1=1/(1+exp(-1*alpha1*(abs(z)-thickness1)));'
                                    'sigmoid2=1/(1+exp(-1*alpha2*(abs(z)-thickness2)))', CustomGBForce.SingleParticle)
    
    #Born Radius
    _ = im_mem.addComputedValue("B", "1/(1/or - tanh(psi + 0.8*psi^2 + 4.85*psi^3)/radius);"
                              "psi=Imol*or/12.56637; or=radius-0.009", CustomGBForce.SingleParticle)
    
    #Add paired polar solvation term term (138.935456 converts e^2/nm to kJ/mol
    _ = im_mem.addEnergyTerm("-0.5*138.935456*(1/soluteDielectric-1/epsPair)*(q1*q2/f);" #Energy Term
                             "epsPair=0.5*(eps1 + eps2);" #How to deal with different eps solvents
                             "f=sqrt(r^2+B1*B2*exp(-r^2/(4*B1*B2)))", CustomGBForce.ParticlePairNoExclusions) #fGB

    #Add self polar solvation term i=j
    _ = im_mem.addEnergyTerm('-0.5*138.935456*(1/soluteDielectric-1/eps)*q^2/B', CustomGBForce.SingleParticle)
    
    #Self Solvent Burial (Esa*4*pi=28.3919551)
    _ = im_mem.addEnergyTerm("28.3919551*((radius+probe)^2)*((radius/B)^6);"
                             'probe=0.14 + 0.1*(1-tanh((abs(z) - thickness2)/(0.25)))', CustomGBForce.SingleParticle)

    # Copy charges and radii from NonbondedForce into GBSA
    nbf = [force for force in system.getForces() if type(force) == NonbondedForce][0]
    for index in range(nbf.getNumParticles()):
        charge, sigma, epsilon = nbf.getParticleParameters(index)
        radius = 0.4 * sigma  # Extremely Rough better to get from a forcefield
        #scaling_factor = 1.0            # OBC scaling (often 1.0)
        #Hmmmmmmm OBC assigns much closer all of them to 0.1
        scaling_factor = 0.1
        im_mem.addParticle((charge, radius, scaling_factor))
    
    im_mem.setNonbondedMethod(nonbonded_method)
    if nonbonded_cutoff:
        im_mem.setCutoffDistance(nonbonded_cutoff)

    return im_mem

In [4]:
def simulate_from_filepair(pdb_top_fn, sys_xml_fn,
                           temp=300, ts=0.001, n_steps=1e5, #100ps
                           dcd_fn='output.dcd', dcd_freq=1000,
                           stdout_fn='output.stdout', stdout_freq=1000,
                           working_dir='./'):
    if not os.path.isdir(working_dir):
        os.makedirs(working_dir, exist_ok=True)
    #Construct
    pdb = PDBFile(pdb_top_fn)
    with open(sys_xml_fn, 'r') as f:
        system = XmlSerializer.deserialize(f.read())
    integrator = LangevinMiddleIntegrator(temp*kelvin, 1/picosecond, ts*picoseconds)
    sim = Simulation(pdb.topology, system, integrator)
    _ = sim.context.setPositions(pdb.positions)
    #Minimize
    _ = sim.minimizeEnergy()
    with open(os.path.join(working_dir,'minimized.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    #Simulate
    _ = sim.reporters.append(DCDReporter(os.path.join(working_dir, dcd_fn), dcd_freq))
    _ = sim.reporters.append(StateDataReporter(os.path.join(working_dir, stdout_fn), stdout_freq, step=True, potentialEnergy=True, temperature=True, speed=True))
    _ = sim.step(n_steps)

    #Final
    state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
    contents = XmlSerializer.serialize(state)
    with open(os.path.join(working_dir, 'final_state.xml'), 'w') as f:
        _ = f.write(contents)
    with open(os.path.join(working_dir,'final_top.pdb'), 'w') as f:
        _ = PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    return None

In [8]:
print('Parameterizing...')
sys, top, pos = parameterize_from_pdb('6drx_bp_PL.pdb',
                                      nonbonded_method=CutoffNonPeriodic,
                                      nonbonded_cutoff=2)
print('Parameterizing Done')
im_mem_force = implicit_membrane_force(system=sys,
                                       nonbonded_method=CustomGBForce.CutoffNonPeriodic,
                                       nonbonded_cutoff=Quantity(value=2, unit=nanometer))
print('Membrane Force Made')
sys.addForce(im_mem_force)

5756


/home/exouser/miniconda3/envs/prep/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


01/15/2026 23:38:31//Saved prepared ligand to ./dummy_L.pdb ./dummy_L.sdf


[23:38:31] WARNING: More than one matching pattern found - picking one



5756
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737b7c1b4db0> >
1 5756


/media/volume/Josephs-Volume/githubs/Bridgeport/ForceFields/ForceFieldHandler.py:158: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  positions = np.array(self.interchange.positions) * nanometer


5

In [12]:
sys.getForce(5).getParticleParameters(69)

(0.4196, 0.04276313847073629, 0.1)

In [8]:
top.getNumAtoms(), sys.getNumParticles()

(5756, 5756)

In [11]:
working_dir = './implicit_pore'
if not os.path.isdir(working_dir):
    os.makedirs(working_dir, exist_ok=True)

In [9]:
xml_fn = 'implicit_pore/implicit_pore_PL_gas.xml'
with open(xml_fn, 'w') as f:
    f.write(XmlSerializer.serialize(sys))

In [12]:
simulate_from_filepair('6drx_bp_PL.pdb', 'implicit_pore/implicit_pore_PL_gas.xml',
                       temp=300, ts=0.001, n_steps=1e5,
                       dcd_fn='output.dcd', dcd_freq=1000,
                       stdout_fn='output.stdout', stdout_freq=1000,
                       working_dir=working_dir)

In [ ]:
raise Exception('Purposeful Exception in order to halt restart run all here')

In [13]:
gas_sys, gas_top, gas_pos = parameterize_from_pdb('6drx_bp_PL.pdb',
                                                  nonbonded_method=CutoffNonPeriodic,
                                                  nonbonded_cutoff=2)

mem_sys, mem_top, mem_pos = parameterize_from_pdb('6drx_bp_PL.pdb',
                                                  nonbonded_method=CutoffNonPeriodic,
                                                  nonbonded_cutoff=2)

obc_sys, obc_top, obc_pos = parameterize_from_pdb('6drx_bp_PL.pdb',
                                                  nonbonded_method=CutoffNonPeriodic,
                                                  nonbonded_cutoff=2, build_w_obc2=True)

im_mem_force = implicit_membrane_force(system=mem_sys,
                                       nonbonded_method=CustomGBForce.CutoffNonPeriodic,
                                       nonbonded_cutoff=Quantity(value=2, unit=nanometer))
mem_sys.addForce(im_mem_force)

5756


/home/exouser/miniconda3/envs/prep/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


01/15/2026 23:40:24//Saved prepared ligand to ./dummy_L.pdb ./dummy_L.sdf


[23:40:24] WARNING: More than one matching pattern found - picking one



5756
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737c9bd564f0> >
1 5756
5756
01/15/2026 23:40:25//Saved prepared ligand to ./dummy_L.pdb ./dummy_L.sdf


[23:40:25] WARNING: More than one matching pattern found - picking one



5756
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737c9cb57d70> >
1 5756
5756
01/15/2026 23:40:26//Saved prepared ligand to ./dummy_L.pdb ./dummy_L.sdf


[23:40:26] WARNING: More than one matching pattern found - picking one



5756
<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737b7abef5b0> >
1 5756
<openmm.openmm.CustomGBForce; proxy of <Swig Object of type 'OpenMM::CustomGBForce *' at 0x737b7abf57b0> >
1 5705


5

In [14]:
for (sys, top, pos), subdr in zip(((gas_sys, gas_top, gas_pos),
                                   (mem_sys, mem_top, mem_pos),
                                   (obc_sys, obc_top, obc_pos)),
                                  ['Gas', 'Mem', 'OBC']):
    print(sys.getNumForces(), subdr, top)

5 Gas <Topology; 2 chains, 354 residues, 5756 atoms, 5826 bonds>
6 Mem <Topology; 2 chains, 354 residues, 5756 atoms, 5826 bonds>
6 OBC <Topology; 2 chains, 354 residues, 5756 atoms, 5826 bonds>


In [15]:
[froce for froce in gas_sys.getForces()]

[<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x737b7bbdeb70> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x737b7cc5f970> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737b7cc5fd30> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x737b7bbde930> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x737b7cc5fc70> >]

In [16]:
[froce for froce in mem_sys.getForces()]

[<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x737b7cc5e0f0> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x737b7ac0b0f0> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737b7bf7a6f0> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x737b7ac0adf0> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x737b7a8e6230> >,
 <openmm.openmm.CustomGBForce; proxy of <Swig Object of type 'OpenMM::CustomGBForce *' at 0x737b7bbde070> >]

In [17]:
[froce for froce in obc_sys.getForces()]

[<openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x737b7abf6230> >,
 <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x737b7abf6030> >,
 <openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x737b7abf5ab0> >,
 <openmm.openmm.CMMotionRemover; proxy of <Swig Object of type 'OpenMM::CMMotionRemover *' at 0x737b7abf5e30> >,
 <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x737b7abad730> >,
 <openmm.openmm.CustomGBForce; proxy of <Swig Object of type 'OpenMM::CustomGBForce *' at 0x737b7abf5eb0> >]

In [18]:
mem_force, obc_force = mem_sys.getForce(5), obc_sys.getForce(5)

In [19]:
types_in = lambda x: [type(elem) for elem in x]

In [23]:
obc_sys.getForce(5).getNumParticles(), mem_sys.getNumParticles()

(5705, 5756)

In [20]:
for i in range(mem_force.getNumParticles()):
    mem_params = mem_force.getParticleParameters(i)
    obc_params = obc_force.getParticleParameters(i)
    mem_types = types_in(mem_params)
    obc_types = types_in(obc_params)
    if i % 10 == 0:
        print(mem_params)
        print(obc_params)
        print('######################')

(0.0017, 0.12999994095103834, 0.1)
(0.0017, 0.146, 0.11534)
######################
(0.1202, 0.07839908719634987, 0.1)
(0.1202, 0.111, 0.09435)
######################
(0.0136, 0.13598678033694142, 0.1)
(0.0136, 0.16099999999999998, 0.11591999999999998)
######################
(-0.0425, 0.10598131150997477, 0.1)
(-0.0425, 0.111, 0.09435)
######################
(0.085, 0.09885412176485205, 0.1)
(0.085, 0.111, 0.09435)
######################
(-0.0252, 0.13598678033694142, 0.1)
(-0.0252, 0.16099999999999998, 0.11591999999999998)
######################
(0.5973, 0.13598678033694142, 0.1)
(0.5973, 0.16099999999999998, 0.11591999999999998)
######################
(-0.3479, 0.12999994095103834, 0.1)
(-0.3479, 0.146, 0.11534)
######################
(0.1426, 0.09885412176485205, 0.1)
(0.1426, 0.111, 0.09435)
######################
(0.34, 0.04276313847073629, 0.1)
(0.34, 0.12100000000000001, 0.10285000000000001)
######################
(0.2719, 0.04276313847073629, 0.1)
(0.2719, 0.12100000000000001, 0

OpenMMException: Assertion failure at CustomGBForce.cpp:134.  Index out of range

In [39]:
mem_params, mem_types, obc_params, obc_types

((-0.0232, 0.10598131150997477, 1.0),
 [float, float, float],
 (-0.0232, 0.111, 0.09435),
 [float, float, float])

In [54]:
#Dielectrics
epsWat = 78.3
epsMid = 3
epsCore = 1
epsSolute = 1

#Membrane thickness
thickness1 = 1.2 #nanometers
thickness2 = 1.8 #nanometers
pore_radius = 1.0 #nanometers should be the inflection radially

#Functions
alpha = 7.2
sigmoid = lambda x, alpha, thickness: (1)/(1+(np.exp(-1*alpha*(np.abs(x) - thickness))))
epz = lambda z: epsCore + (epsMid-epsCore)*sigmoid(z, alpha, thickness1) + (epsWat - epsCore)*sigmoid(z, alpha, thickness2)
eps_func = lambda x, y, z: epsWat + (epz(z) - epsWat)*sigmoid(np.sqrt(x**2 + y**2), 3*alpha, pore_radius)

In [55]:
domain_points = [-3, -2, -1, 0, 1, 2, 3] #nanometers

In [56]:
import itertools as itr

In [57]:
nx, ny, nz = 121, 121, 121 #n points from -3 to +3 nanometer at 0.5A step

In [58]:
values = np.zeros((nx, ny, nz))

In [59]:
domain = np.linspace(-3, 3, 121)

In [60]:
for i in range(nx):
    for j in range(ny):
        for k in range(nz):
            values[i, j, k] = eps_func(domain[i], domain[j], domain[k])

In [61]:
import numpy as np
from gridData import Grid  # comes from the GridDataFormats package

# --- Your custom volumetric data ---
# shape = (nx, ny, nz)
#data = np.random.random((40, 40, 40)).astype(np.float64)

# Grid origin (Å)
origin = np.array([-30.0, -30.0, -30.0], dtype=float)

# Voxel spacing along x, y, z (Å) — orthogonal grid
delta = np.array([0.5, 0.5, 0.5], dtype=float)



In [62]:
# Build a Grid and export as OpenDX
g = Grid(values, origin=origin, delta=delta)
g.export("my_density.dx", file_format="dx", type="double")

This sigmoidal correction to the pore is complete, these are the dielectric isosurfaces:

| Color       | Dielectric Value |
|-----        |-----             |
| Blue (Dark) | 78.3 (Water)     |
| Beige       | 70               |
| Green       | 50               |
| Cyan        | 20               |
| Pink        | 3                |
| Orange      | 1.01             |


In [5]:
import openmmtools
waterbox_no_pore = openmmtools.testsystems.WaterBox(box_edge=Quantity(60.0, angstrom), ionic_strength=150*millimolar, nonbondedMethod=CutoffPeriodic)
waterbox_ye_pore = openmmtools.testsystems.WaterBox(box_edge=Quantity(60.0, angstrom), ionic_strength=150*millimolar, nonbondedMethod=CutoffPeriodic)

Warning on use of the timeseries module: If the inherent timescales of the system are long compared to those being analyzed, this statistical inefficiency may be an underestimate.  The estimate presumes the use of many statistically independent samples.  Tests should be performed to assess whether this condition is satisfied.   Be cautious in the interpretation of the data.

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



In [6]:
nbf = [force for force in waterbox_no_pore.system.getForces() if type(force) == NonbondedForce][0]
im_mem_no_pore = implicit_membrane_force(waterbox_no_pore.system, nonbonded_cutoff=nbf.getCutoffDistance(), nonbonded_method=nbf.getNonbondedMethod(), water_pore=False)

nbf = [force for force in waterbox_ye_pore.system.getForces() if type(force) == NonbondedForce][0]
im_mem_ye_pore = implicit_membrane_force(waterbox_ye_pore.system, nonbonded_cutoff=nbf.getCutoffDistance(), nonbonded_method=nbf.getNonbondedMethod(), water_pore=True)

In [7]:
waterbox_ye_pore.system.addForce(im_mem_ye_pore), waterbox_no_pore.system.addForce(im_mem_no_pore)

(4, 4)

In [8]:
sim_no_pore = Simulation(waterbox_no_pore.topology, waterbox_no_pore.system, LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 1*femtosecond))

In [9]:
sim_ye_pore = Simulation(waterbox_ye_pore.topology, waterbox_ye_pore.system, LangevinMiddleIntegrator(300*kelvin, 1/picosecond, 1*femtosecond))

In [10]:
for sim, working_dir in zip([sim_ye_pore, sim_no_pore],['waterbox_ye_pore', 'waterbox_no_pore']):
    if not os.path.isdir(working_dir):
        os.makedirs(working_dir, exist_ok=True)
    
    sim.context.setPositions(waterbox_ye_pore.positions)
    sim.minimizeEnergy()
    sim.reporters.append(DCDReporter(os.path.join(working_dir, 'trajectory.dcd'), 5000)) #Every 5 picoseconds
    sim.reporters.append(StateDataReporter(os.path.join(working_dir, 'stdout.stdout'), 1000, #Every picosecond
                                                   step=True, potentialEnergy=True, temperature=True, speed=True))
    sim.step(2e5) #300 picoseconds
    
    state = sim.context.getState(getPositions=True, getVelocities=True, enforcePeriodicBox=True)
    contents = XmlSerializer.serialize(state)
    with open(os.path.join(working_dir, 'final_state.xml'), 'w') as f:
        f.write(contents)
    with open(os.path.join(working_dir,'final_top.pdb'), 'w') as f:
        PDBFile.writeFile(sim.topology, sim.context.getState(getPositions=True).getPositions(), file=f, keepIds=True)
    
    print('Done!')
    

Done!
Done!
